In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
import os
import argparse
from typing import List, Tuple, Dict, Any, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Batch

# Make sure these imports point to your actual project structure
from model.getdata import smiles2graph
from finetune_pair import load_pretrained_encoders, ensure_dir, pct_within_1_np, load_pretrained_encoders, _stack_encode, TempPairMLP, SolvationPredictor, build_model_subset
from model.fusion import WeightedFusion_new as WeightedFusion

[info] Using device: cuda


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# This will be set in the configuration cell
EMB_DIM: int = 512

Using device: cuda


In [3]:
import math
import torch
import torch.nn as nn


class RatioGate(nn.Module):
    """
    Very small ratio gate:

        ratio -> g in [0, ~0.25 * scale]

    We use g ~ x(1-x) so that:
      - g ≈ 0 when ratio ≈ 0 or 1  (no interaction)
      - g is maximal around ratio ≈ 0.5 (strongest interaction)

    The constructor keeps (ratio_dim, hidden_dim) only for API
    compatibility with the old version.
    """

    def __init__(self, ratio_dim: int = 1, hidden_dim: int = 16):
        super().__init__()
        # Single learnable scale; you can initialize it smaller if needed.
        self.scale = nn.Parameter(torch.tensor(1.0))

    def forward(self, ratio: torch.Tensor) -> torch.Tensor:
        """
        ratio: (B,) or (B,1)
        returns:
            g: (B,1)
        """
        if ratio.dim() == 1:
            ratio = ratio.unsqueeze(-1)      # (B,1)

        # Clamp to [0,1] just in case
        x = torch.clamp(ratio, 0.0, 1.0)     # (B,1)

        base = x * (1.0 - x)                 # x(1-x), endpoints -> 0
        g = self.scale * base                # (B,1)
        return g


class RSCrossAttention(nn.Module):
    """
    Simplified Ratio-aware Symmetric Cross-Attention.

    Differences from the old version:
      - No Q/K/V projection layers; tokens are used directly as Q/K/V.
      - Always uses softmax (no entmax).
      - Very small RatioGate.

    API is kept the same:
      __init__(d_model, use_entmax, ratio_dim, ratio_hidden_dim)
      forward(tokens_a, tokens_b, ratio)
    """

    def __init__(
        self,
        d_model: int,
        use_entmax: bool = False,   # kept for compatibility, not used
        ratio_dim: int = 1,
        ratio_hidden_dim: int = 16,
    ):
        super().__init__()
        self.d_model = d_model
        self.scale = math.sqrt(d_model)

        # Small ratio gate
        self.ratio_gate = RatioGate(ratio_dim=ratio_dim,
                                    hidden_dim=ratio_hidden_dim)

    def _cross_once(
        self,
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
    ):
        """
        One-way cross attention.

        q: (B, Lq, D)
        k: (B, Lk, D)
        v: (B, Lk, D)

        returns:
            out:  (B, Lq, D)
            attn: (B, Lq, Lk)
        """
        # (B, Lq, Lk)
        scores = torch.matmul(q, k.transpose(-2, -1)) / self.scale

        # Softmax over key dimension
        attn = torch.softmax(scores, dim=-1)          # (B, Lq, Lk)

        # (B, Lq, D)
        out = torch.matmul(attn, v)
        return out, attn

    def forward(
        self,
        tokens_a: torch.Tensor,
        tokens_b: torch.Tensor,
        ratio: torch.Tensor,
    ):
        """
        tokens_a: (B, La, D)  e.g. solvent1 ChemSpace tokens
        tokens_b: (B, Lb, D)  e.g. solvent2 ChemSpace tokens
        ratio:    (B,) or (B,1)  Xsolvent1

        returns:
            out_a:    (B, La, D)
            out_b:    (B, Lb, D)
            attn_a2b: (B, La, Lb)
            attn_b2a: (B, Lb, La)
            g:        (B,1)
        """
        B, La, D = tokens_a.shape
        B2, Lb, D2 = tokens_b.shape
        assert B == B2, "batch size mismatch between A and B"
        assert D == D2 == self.d_model, "embedding dim mismatch"

        # Use tokens directly as Q/K/V
        Qa, Ka, Va = tokens_a, tokens_a, tokens_a
        Qb, Kb, Vb = tokens_b, tokens_b, tokens_b

        # A attends to B
        cross_a, attn_a2b = self._cross_once(Qa, Kb, Vb)   # (B,La,D), (B,La,Lb)
        # B attends to A
        cross_b, attn_b2a = self._cross_once(Qb, Ka, Va)   # (B,Lb,D), (B,Lb,La)

        # ratio -> scalar gate g
        g = self.ratio_gate(ratio)                         # (B,1)
        g_expanded = g.view(B, 1, 1)                       # (B,1,1), broadcast

        # Residual interpolation:
        # when ratio ≈ 0/1, g ≈ 0, so out ≈ original tokens (almost no interaction)
        out_a = (1.0 - g_expanded) * tokens_a + g_expanded * cross_a
        out_b = (1.0 - g_expanded) * tokens_b + g_expanded * cross_b

        return out_a, out_b, attn_a2b, attn_b2a, g


In [4]:
import torch
import torch.nn as nn
from torch_geometric.data import Batch
from typing import List, Optional

class BMixSolvationPredictor(nn.Module):
    def __init__(
        self,
        sol_encs: List[nn.Module],
        solv_encs: List[nn.Module],
        sol_fusion: nn.Module,
        solv_fusion: nn.Module,
        combiner: TempPairMLP,
        d_model: int = EMB_DIM,
        use_entmax: bool = False,
        ratio_dim: int = 1,
        ratio_hidden_dim: int = 16,
        # NEW: whether to use attention in the residual (M_eff)
        use_attn_for_residual: bool = True,
    ):
        super().__init__()
        self.sol_encs = nn.ModuleList(sol_encs)
        self.solv_encs = nn.ModuleList(solv_encs)
        self.sol_f = sol_fusion
        self.solv_f = solv_fusion
        self.combine = combiner

        # Cross-attention over ChemSpace tokens (can be turned off in residual)
        self.attends = RSCrossAttention(
            d_model=d_model,
            use_entmax=use_entmax,
            ratio_dim=ratio_dim,
            ratio_hidden_dim=ratio_hidden_dim,
        )

        # Global scale for the scalar residual Δy (always positive)
        self.cross_scale = nn.Parameter(torch.tensor(0.0))

        # Per-space scalar heads: z2_0[k,:] -> a_k, z3_0[k,:] -> b_k
        self.space_head_s1 = nn.Linear(d_model, 1)
        self.space_head_s2 = nn.Linear(d_model, 1)

        # NEW: mode flag
        self.use_attn_for_residual = use_attn_for_residual

    def _fuse(self, fusion: nn.Module, embs: torch.Tensor, g: Optional[torch.Tensor]):
        """
        Fusion helper; handles the (out, aux) tuple case of WeightedFusion.
        """
        out = fusion(embs, global_features=g)
        return out[0] if isinstance(out, tuple) else out

    def forward(
        self,
        sol_batch: Batch,
        solv1_batch: Batch,
        solv2_batch: Batch,
        ratio_batch: torch.Tensor,
        temps: torch.Tensor,
    ):
        """
        sol_batch  : Batch of solute graphs
        solv1_batch: Batch of solvent1 graphs
        solv2_batch: Batch of solvent2 graphs
        ratio_batch: (B,) or (B,1), mole fraction of solvent1
        temps      : (B,1) temperature

        Returns:
            pred : (B,1) final prediction (ideal + residual)
            z1   : (B,D) solute embedding
            zv   : (B,D) ideal mixed solvent embedding
            M    : (B,K,K) interaction matrix used in the residual
                   - light mode:   M = base_M = a⊗b
                   - full  mode:   M = M_eff = attn ⊙ (a⊗b)
        """
        # ---- 1. Encode solute / two solvents in ChemSpace ----
        # Per-space embeddings: [B, K, D]
        z1_spaces = _stack_encode(self.sol_encs, sol_batch)
        z2_0 = _stack_encode(self.solv_encs, solv1_batch)
        z3_0 = _stack_encode(self.solv_encs, solv2_batch)

        # Optional global features
        g1 = getattr(sol_batch, "global_features", None)
        g2 = getattr(solv1_batch, "global_features", None)
        g3 = getattr(solv2_batch, "global_features", None)

        # Fused embeddings: [B, D]
        z1 = self._fuse(self.sol_f, z1_spaces, g1)
        z2 = self._fuse(self.solv_f, z2_0, g2)
        z3 = self._fuse(self.solv_f, z3_0, g3)

        # ---- 2. Ideal mixing in embedding space ----
        ratio = ratio_batch.view(-1, 1)          # (B,1)
        z_lin = ratio * z2 + (1.0 - ratio) * z3  # (B,D)
        zv = z_lin                               # ideal mixed solvent embedding

        # First-pass prediction = ideal mixing prediction
        y_ideal = self.combine(z1, zv, temps)    # (B,) or (B,1)
        if y_ideal.dim() == 1:
            y_ideal = y_ideal.view(-1, 1)        # (B,1)

        # ---- 3. ChemSpace×ChemSpace interaction matrix ----
        # Per-space scalars a_k, b_k from original per-space embeddings
        # a, b: [B,K]
        a = self.space_head_s1(z2_0).squeeze(-1)  # solvent1
        b = self.space_head_s2(z3_0).squeeze(-1)  # solvent2

        # Outer product base_M[b, i, j] = a[b,i] * b[b,j], shape [B,K,K]
        base_M = a.unsqueeze(-1) * b.unsqueeze(-2)   # [B,K,K]

        if self.use_attn_for_residual:
            # Cross-attention over ChemSpaces (solvent1 -> solvent2)
            # We only need attn_a2b here for re-weighting
            _, _, attn_a2b, _, g = self.attends(z2_0, z3_0, ratio_batch)
            # Effective interaction matrix: ChemSpace-aware + ratio-aware
            B = base_M.size(0)
            g_expanded = g.view(B, 1, 1)
            M = attn_a2b * base_M * g_expanded                # [B,K,K]
        else:
            # Light mode: no attention, only factorized interaction
            M = base_M                                # [B,K,K]

        # ---- 4. Reduce to a scalar residual Δy for each sample ----
        lambda_cross = torch.exp(self.cross_scale)     # > 0
        # Sum over ChemSpace×ChemSpace -> [B,1]
        dy = lambda_cross * M.sum(dim=(-2, -1), keepdim=True)
        dy = dy.view(dy.size(0), 1) 

        # ---- 5. Final prediction = ideal mixing + residual ----
        pred = y_ideal + dy                            # (B,1)

        return pred, z1, zv, M


In [5]:
from typing import Tuple, List
import torch
from torch_geometric.data import Batch

def build_mix_model_subset(
    sample_s,
    sample_v,
    ckpt_paths: List[str],
    solute_dropout: float,
    solvent_dropout: float,
    mlp_hidden: Tuple[int, ...],
    mlp_dropout: float,
    t_embed_dim: int,
    use_entmax: bool = False,
    ratio_dim: int = 1,
    ratio_hidden_dim: int = 16,
    use_attention: bool = True,  # NEW: light/full mode switch
) -> BMixSolvationPredictor:
    """
    Build a BMixSolvationPredictor using a subset of pre-trained encoders.

    Parameters
    ----------
    sample_s, sample_v : Data
        Example solute and solvent graphs used to infer feature sizes.
    ckpt_paths : list of str
        Paths to encoder checkpoints (one per ChemSpace task).
    solute_dropout, solvent_dropout : float
        Dropout rates for solute / solvent fusion modules.
    mlp_hidden : tuple of int
        Hidden layer sizes for TempPairMLP.
    mlp_dropout : float
        Dropout rate in TempPairMLP.
    t_embed_dim : int
        Temperature embedding dimension.
    use_entmax : bool, optional
        Whether to use entmax instead of softmax in RSCrossAttention.
    ratio_dim : int, optional
        Dimensionality of ratio features fed into RSCrossAttention (e.g. 1 for x_A).
    ratio_hidden_dim : int, optional
        Hidden size of the ratio-gating MLP in RSCrossAttention.
    use_attn_for_residual : bool, optional
        If True, the residual Δy uses the attention-weighted interaction
        matrix M_eff = attn ⊙ (a ⊗ b).
        If False, it uses the factorized base_M = a ⊗ b only (light mode).

    Returns
    -------
    BMixSolvationPredictor
    """
    # 1) Load pre-trained encoders
    sol_encs = load_pretrained_encoders(sample_s, ckpt_paths)
    solv_encs = load_pretrained_encoders(sample_v, ckpt_paths)

    # 2) Build WeightedFusion modules for solute and solvent encoders
    sol_f = WeightedFusion(
        len(ckpt_paths),
        EMB_DIM,
        dropout=solute_dropout,
        pair=True,
    ).to(device)

    solv_f = WeightedFusion(
        len(ckpt_paths),
        EMB_DIM,
        dropout=solvent_dropout,
        pair=True,
    ).to(device)

    # 3) Probe the fused embedding dimension z_dim (for TempPairMLP)
    with torch.no_grad():
        b1 = Batch.from_data_list([sample_s]).to(device)
        b2 = Batch.from_data_list([sample_v]).to(device)

        z1 = _stack_encode(sol_encs, b1)  # (1, K_sol, EMB_DIM)
        z2 = _stack_encode(solv_encs, b2) # (1, K_solv, EMB_DIM)

        g1 = getattr(b1, "global_features", None)
        g2 = getattr(b2, "global_features", None)

        z1 = sol_f(z1, g1)
        z2 = solv_f(z2, g2)

        if isinstance(z1, tuple):
            z1 = z1[0]
        if isinstance(z2, tuple):
            z2 = z2[0]

        z_dim = z1.shape[-1]  # fused embedding dim for TempPairMLP

    # 4) Build MLP head for temperature + (solute, mixed solvent)
    comb = TempPairMLP(
        z_dim,
        t_embed_dim=t_embed_dim,
        hidden=mlp_hidden,
        dropout=mlp_dropout,
    ).to(device)

    # 5) Build BMixSolvationPredictor
    model = BMixSolvationPredictor(
        sol_encs=sol_encs,
        solv_encs=solv_encs,
        sol_fusion=sol_f,
        solv_fusion=solv_f,
        combiner=comb,
        d_model=EMB_DIM,           # per-space dim
        use_entmax=use_entmax,
        ratio_dim=ratio_dim,
        ratio_hidden_dim=ratio_hidden_dim,
        use_attn_for_residual=use_attention,  # pass the flag
    ).to(device)

    return model


In [6]:
from typing import List, Tuple, Optional
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from torch_geometric.data import Batch

from model.getdata import smiles2graph  


def _smiles_graph_inputs_from_mixture_csv(csv_path: str):
    """
    Adapted for "mixed solvent" CSV:

    solute, solvent1, solvent2, Xsolvent2, temperature, Xsolvent1, s,
    solute_smiles, solvent1_smiles, solvent2_smiles, logS

    Notes:
        - *_smiles columns are used for graph construction
        - Xsolvent1 / Xsolvent2 are mole fractions of the two solvents
        - logS is the label (already log10 S)

    Returns:
        d1, d2_1, d2_2 : (PyG Data list)      # solute, solvent1, solvent2 graphs
        y             : (N,) float32         # logS
        T             : (N,) float32         # temperature
        frac1, frac2  : (N,) float32         # Xsolvent1, Xsolvent2
        solute_smiles, solv1_smiles, solv2_smiles : List[str]
    """
    df = pd.read_csv(csv_path)

    # Use *_smiles columns explicitly, not name columns
    required = [
        "solute_smiles",
        "solvent1_smiles",
        "solvent2_smiles",
        "temperature",
        "logS",
        "Xsolvent1",
        "Xsolvent2",
    ]
    for col in required:
        if col not in df.columns:
            raise ValueError(f"Required column '{col}' not found in {csv_path}")

    solute_smiles = df["solute_smiles"].astype(str).tolist()
    solv1_smiles = df["solvent1_smiles"].astype(str).tolist()
    solv2_smiles = df["solvent2_smiles"].astype(str).tolist()

    y = df["logS"].astype(float).to_numpy()
    T = df["temperature"].astype(float).to_numpy()
    frac1 = df["Xsolvent1"].astype(float).to_numpy()
    frac2 = df["Xsolvent2"].astype(float).to_numpy()

    # Use smiles2graph for graph construction
    # y / properties are just attached to Data, actual training uses collate_mixture for labels / temps
    d1 = smiles2graph(solute_smiles, y=y, properties=T)
    d2_1 = smiles2graph(solv1_smiles, y=y, properties=T)
    d2_2 = smiles2graph(solv2_smiles, y=y, properties=T)

    return d1, d2_1, d2_2, y, T, frac1, frac2, solute_smiles, solv1_smiles, solv2_smiles


In [7]:
class MixtureDataset(Dataset):
    """
    Dataset for solvation in mixed solvents.
        Each sample returns:
        (solute_graph, solvent1_graph, solvent2_graph,
         temperature, label, frac1, frac2,
         solute_smiles, solv1_smiles, solv2_smiles)
    """
    def __init__(
        self,
        s,
        v1,
        v2,
        temps,
        labels,
        frac1,
        frac2,
        solute_smiles,
        solv1_smiles,
        solv2_smiles,
    ):
        n = len(labels)
        assert len(s) == len(v1) == len(v2) == len(temps) == len(frac1) == len(frac2) == n
        assert len(solute_smiles) == len(solv1_smiles) == len(solv2_smiles) == n

        self.s = s
        self.v1 = v1
        self.v2 = v2
        self.temps = temps
        self.labels = labels
        self.frac1 = frac1
        self.frac2 = frac2
        self.solute_smiles = solute_smiles
        self.solv1_smiles = solv1_smiles
        self.solv2_smiles = solv2_smiles

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.s[idx],
            self.v1[idx],
            self.v2[idx],
            float(self.temps[idx]),
            float(self.labels[idx]),
            float(self.frac1[idx]),
            float(self.frac2[idx]),
            self.solute_smiles[idx],
            self.solv1_smiles[idx],
            self.solv2_smiles[idx],
        )


def collate_mixture(batch):
    """
    Collate function for mixture data.

    returns:
        b1, b2, b3         : Batch (solute, solvent1, solvent2)
        temps_tensor       : (B,1)
        labels_tensor      : (B,1)
        w1, w2             : (B,1)  Xsolvent1 / Xsolvent2
        solute_smiles_list,
        solv1_smiles_list,
        solv2_smiles_list
    """
    (
        sol,
        solv1,
        solv2,
        temps,
        labels,
        frac1,
        frac2,
        solute_smiles,
        solv1_smiles,
        solv2_smiles,
    ) = zip(*batch)

    b1 = Batch.from_data_list(list(sol))
    b2 = Batch.from_data_list(list(solv1))
    b3 = Batch.from_data_list(list(solv2))

    temps = torch.tensor(temps, dtype=torch.float32).view(-1, 1)
    labels = torch.tensor(labels, dtype=torch.float32).view(-1, 1)
    w1 = torch.tensor(frac1, dtype=torch.float32).view(-1, 1)
    w2 = torch.tensor(frac2, dtype=torch.float32).view(-1, 1)

    solute_smiles = list(solute_smiles)
    solv1_smiles = list(solv1_smiles)
    solv2_smiles = list(solv2_smiles)

    return (
        b1,
        b2,
        b3,
        temps,
        labels,
        w1,
        w2,
        solute_smiles,
        solv1_smiles,
        solv2_smiles,
    )
    # Example usage:
    # dataset = MixtureDataset(s, v1, v2, temps, labels, frac1, frac2, solute_smiles, solv1_smiles, solv2_smiles)
    # loader = DataLoader(dataset, batch_size=32, collate_fn=collate_mixture, shuffle=True)

In [8]:
def _split_indices_by_solvent_pair(
    solv1_smiles: List[str],
    solv2_smiles: List[str],
    train_frac: float,
    val_frac: float,
    test_frac: float,
    random_state: int = 42,
    canonicalize_pair: bool = True,
):
    """
    根据 (solvent1, solvent2) 组合划分索引集合。

    canonicalize_pair=True 时，把 (A,B) 和 (B,A) 视为同一种溶剂对：
        key = tuple(sorted((s1, s2)))
    """
    assert abs(train_frac + val_frac - 1.0 + test_frac) < 1e-6 or \
           abs(train_frac + val_frac + test_frac - 1.0) < 1e-6, \
        "train/val/test 比例之和必须为 1"

    n = len(solv1_smiles)
    assert len(solv2_smiles) == n

    keys = []
    for s1, s2 in zip(solv1_smiles, solv2_smiles):
        if canonicalize_pair:
            key = tuple(sorted((str(s1), str(s2))))
        else:
            key = (str(s1), str(s2))
        keys.append(key)

    unique_pairs = sorted(set(keys))
    rng = np.random.RandomState(random_state)
    rng.shuffle(unique_pairs)

    n_pairs = len(unique_pairs)
    n_train = int(round(train_frac * n_pairs))
    n_val = int(round(val_frac * n_pairs))
    # 剩下的全部给 test，避免因为 round 导致和不为 1
    n_test = max(0, n_pairs - n_train - n_val)

    train_pairs = set(unique_pairs[:n_train])
    val_pairs = set(unique_pairs[n_train:n_train + n_val])
    test_pairs = set(unique_pairs[n_train + n_val:])

    idx_train, idx_val, idx_test = [], [], []
    for i, key in enumerate(keys):
        if key in train_pairs:
            idx_train.append(i)
        elif key in val_pairs:
            idx_val.append(i)
        else:
            idx_test.append(i)

    return idx_train, idx_val, idx_test


In [9]:
def make_mixture_loaders_by_solvent_pair(
    csv_path: str,
    batch_size: int,
    train_frac: float = 0.8,
    val_frac: float = 0.1,
    test_frac: float = 0.1,
    random_state: int = 42,
    canonicalize_pair: bool = True,
):
    """
    从一个 *混合溶剂* CSV（表头如截图）构造
    train / val / test 三个 DataLoader，
    划分依据是 (solvent1_smiles, solvent2_smiles) 组合。

    canonicalize_pair=True:
        (A,B) 和 (B,A) 视作同一种溶剂对，一起划到同一个集合。
    """
    (
        d1,
        d2_1,
        d2_2,
        y,
        T,
        frac1,
        frac2,
        solute_smiles,
        solv1_smiles,
        solv2_smiles,
    ) = _smiles_graph_inputs_from_mixture_csv(csv_path)

    dataset = MixtureDataset(
        d1,
        d2_1,
        d2_2,
        T,
        y,
        frac1,
        frac2,
        solute_smiles,
        solv1_smiles,
        solv2_smiles,
    )

    idx_train, idx_val, idx_test = _split_indices_by_solvent_pair(
        solv1_smiles,
        solv2_smiles,
        train_frac=train_frac,
        val_frac=val_frac,
        test_frac=test_frac,
        random_state=random_state,
        canonicalize_pair=canonicalize_pair,
    )

    train_loader = DataLoader(
        Subset(dataset, idx_train),
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_mixture,
    )
    val_loader = DataLoader(
        Subset(dataset, idx_val),
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_mixture,
    )
    test_loader = DataLoader(
        Subset(dataset, idx_test),
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_mixture,
    )

    return train_loader, val_loader, test_loader


In [10]:
import torch

# 加载状态字典
checkpoint = torch.load('ckpt/leeds.pth')
print(type(checkpoint))
# 打印所有键名，了解模型结构
print(checkpoint.keys())
# 打印键的数量
print(len(checkpoint.keys()))

# 假设状态字典在 'model_state_dict' 键下
if 'model_state_dict' in checkpoint:
    state_dict = checkpoint['model_state_dict']
else:
    # 如果加载的文件本身就是状态字典（如之前示例），则直接使用
    state_dict = checkpoint 

# 现在，像之前那样对 state_dict 进行分析
for key in state_dict.keys():
    print(key)

<class 'dict'>
dict_keys(['model_state_dict', 'best_val_rmse'])
2
solute_encs.0.mpn.encoder.cached_zero_vector
solute_encs.0.mpn.encoder.W_i.weight
solute_encs.0.mpn.encoder.W_i.bias
solute_encs.0.mpn.encoder.W_h.weight
solute_encs.0.mpn.encoder.W_h.bias
solute_encs.0.mpn.encoder.W_o.weight
solute_encs.0.mpn.encoder.W_o.bias
solute_encs.0.projection_head.0.weight
solute_encs.0.projection_head.0.bias
solute_encs.0.projection_head.3.weight
solute_encs.0.projection_head.3.bias
solute_encs.1.mpn.encoder.cached_zero_vector
solute_encs.1.mpn.encoder.W_i.weight
solute_encs.1.mpn.encoder.W_i.bias
solute_encs.1.mpn.encoder.W_h.weight
solute_encs.1.mpn.encoder.W_h.bias
solute_encs.1.mpn.encoder.W_o.weight
solute_encs.1.mpn.encoder.W_o.bias
solute_encs.1.projection_head.0.weight
solute_encs.1.projection_head.0.bias
solute_encs.1.projection_head.3.weight
solute_encs.1.projection_head.3.bias
solute_encs.2.mpn.encoder.cached_zero_vector
solute_encs.2.mpn.encoder.W_i.weight
solute_encs.2.mpn.encoder.

C:\Users\27476\AppData\Local\Temp\ipykernel_30148\3539557939.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('ckpt/leeds.pth')


In [11]:
import torch
from typing import List

def load_pretrained_into_bmix(
    model: BMixSolvationPredictor,
    ckpt_path: str,
    device: str = "cpu",
    freeze_encoders: bool = True,
    freeze_fusions: bool = True,
    freeze_combiner: bool = True,
) -> BMixSolvationPredictor:
    """
    Load parts of BMixSolvationPredictor (encoder + fusion [+ combiner optionally])
    from a single-solvent checkpoint (e.g. ckpt/leeds.pth).

    Keys in checkpoint:
        solute_encs.*, solvent_encs.*,
        solute_fusion.*, solvent_fusion.*,
        combiner.*

    Corresponding keys in BMix model:
        sol_encs.*,  solv_encs.*,
        sol_f.*,     solv_f.*,
        combine.*   (optional)

    Parameters
    ----------
    freeze_encoders : whether to freeze sol_encs / solv_encs
    freeze_fusions  : whether to freeze sol_f / solv_f
    freeze_combiner : whether to freeze combine (TempPairMLP)

    Returns
    -------
    model : BMixSolvationPredictor with loaded weights and frozen parameters as specified
    """
    ckpt = torch.load(ckpt_path, map_location=device)
    state = ckpt.get("model_state_dict", ckpt)

    remapped_state = {}

    for k, v in state.items():
        new_k = None

        if k.startswith("solute_encs."):
            new_k = k.replace("solute_encs", "sol_encs", 1)
        elif k.startswith("solvent_encs."):
            new_k = k.replace("solvent_encs", "solv_encs", 1)
        elif k.startswith("solute_fusion."):
            new_k = k.replace("solute_fusion", "sol_f", 1)
        elif k.startswith("solvent_fusion."):
            new_k = k.replace("solvent_fusion", "solv_f", 1)
        elif k.startswith("combiner."):
            new_k = k.replace("combiner", "combine", 1)

        if new_k is not None:
            remapped_state[new_k] = v

    missing, unexpected = model.load_state_dict(remapped_state, strict=False)
    print(f"[load_pretrained_into_bmix] loaded from {ckpt_path}")
    print("  missing keys in model (expected, e.g. attends, cross_scale):")
    print("   ", missing)
    print("  unexpected keys (if any):")
    print("   ", unexpected)

    freeze_prefixes: List[str] = []
    if freeze_encoders:
        freeze_prefixes += ["sol_encs.", "solv_encs."]
    if freeze_fusions:
        freeze_prefixes += ["sol_f.", "solv_f."]
    if freeze_combiner:
        freeze_prefixes += ["combine."]

    n_frozen = 0
    for name, param in model.named_parameters():
        if any(name.startswith(p) for p in freeze_prefixes):
            param.requires_grad = False
            n_frozen += 1

    print(f"[load_pretrained_into_bmix] frozen {n_frozen} parameters.")
    return model


In [12]:
train_loader, val_loader, test_loader = make_mixture_loaders_by_solvent_pair(
    "mix_noaq.csv",
    batch_size=64,
    train_frac=0.3,
    val_frac=0.1,
    test_frac=0.6,
    random_state=2025,
)

In [13]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
import matplotlib.pyplot as plt
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def train_and_eval_mix_model(
    model,
    train_loader,
    val_loader,
    test_loader,
    num_epochs: int,
    lr: float,
    out_dir: str,
):
    ensure_dir(out_dir)
    plots_dir = os.path.join(out_dir, "plots")
    m_dir = os.path.join(out_dir, "M_heatmaps")  # 改成存 M 的热图
    ensure_dir(plots_dir)
    ensure_dir(m_dir)

    model = model.to(device)
    optimizer = Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    train_losses = []
    val_losses = []
    best_val_loss = float("inf")
    best_state = None

    # ===================== 训练 + 验证 =====================
    for epoch in range(1, num_epochs + 1):
        # ----- train -----
        model.train()
        running_train_loss = 0.0
        n_train_samples = 0

        for (
            sol_b,
            solv1_b,
            solv2_b,
            temps,
            labels,
            w1,
            w2,
            solute_smiles,
            solv1_smiles,
            solv2_smiles,
        ) in train_loader:
            sol_b = sol_b.to(device)
            solv1_b = solv1_b.to(device)
            solv2_b = solv2_b.to(device)
            temps = temps.to(device)
            labels = labels.to(device)
            ratio = w1.to(device)  # Xsolvent1 作为 ratio

            optimizer.zero_grad()
            # 新的 forward：pred, z1, zv, M
            pred, z1, zv, M = model(sol_b, solv1_b, solv2_b, ratio, temps)

            # 确保形状兼容 (B,1)
            if pred.dim() == 1:
                pred = pred.view(-1, 1)

            loss = criterion(pred, labels)
            loss.backward()
            optimizer.step()

            running_train_loss += loss.item() * labels.size(0)
            n_train_samples += labels.size(0)

        epoch_train_loss = running_train_loss / n_train_samples
        train_losses.append(epoch_train_loss)

        # ----- val -----
        model.eval()
        running_val_loss = 0.0
        n_val_samples = 0

        with torch.no_grad():
            for (
                sol_b,
                solv1_b,
                solv2_b,
                temps,
                labels,
                w1,
                w2,
                solute_smiles,
                solv1_smiles,
                solv2_smiles,
            ) in val_loader:
                sol_b = sol_b.to(device)
                solv1_b = solv1_b.to(device)
                solv2_b = solv2_b.to(device)
                temps = temps.to(device)
                labels = labels.to(device)
                ratio = w1.to(device)

                pred, z1, zv, M = model(sol_b, solv1_b, solv2_b, ratio, temps)
                if pred.dim() == 1:
                    pred = pred.view(-1, 1)

                loss = criterion(pred, labels)

                running_val_loss += loss.item() * labels.size(0)
                n_val_samples += labels.size(0)

        epoch_val_loss = running_val_loss / n_val_samples
        val_losses.append(epoch_val_loss)

        print(
            f"Epoch {epoch:03d} | "
            f"Train MSE: {epoch_train_loss:.4f} | "
            f"Val MSE: {epoch_val_loss:.4f}"
        )

        # 记录最优 val
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_state = model.state_dict()

    # 保存训练曲线
    plt.figure()
    plt.plot(range(1, num_epochs + 1), train_losses, label="Train MSE")
    plt.plot(range(1, num_epochs + 1), val_losses, label="Val MSE")
    plt.xlabel("Epoch")
    plt.ylabel("MSE loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "train_val_loss_curve.png"), dpi=300)
    plt.close()

    # 加载最优模型
    if best_state is not None:
        model.load_state_dict(best_state)

    # ===================== 测试集评估 =====================
    model.eval()
    all_preds = []
    all_labels = []
    all_temps = []
    all_w1 = []
    all_w2 = []
    all_solute_smiles = []
    all_solv1_smiles = []
    all_solv2_smiles = []

    # 用来画每对溶剂在不同比例下的 M 热力图
    # M_dict[(solv1_smiles, solv2_smiles)][ratio] -> list of M matrices (4x4)
    M_dict = {}

    with torch.no_grad():
        for (
            sol_b,
            solv1_b,
            solv2_b,
            temps,
            labels,
            w1,
            w2,
            solute_smiles,
            solv1_smiles,
            solv2_smiles,
        ) in test_loader:
            sol_b = sol_b.to(device)
            solv1_b = solv1_b.to(device)
            solv2_b = solv2_b.to(device)
            temps_gpu = temps.to(device)
            labels_gpu = labels.to(device)
            ratio = w1.to(device)

            pred, z1, zv, M = model(sol_b, solv1_b, solv2_b, ratio, temps_gpu)
            if pred.dim() == 1:
                pred = pred.view(-1, 1)

            # 收集预测
            all_preds.append(pred.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            all_temps.append(temps.cpu().numpy())
            all_w1.append(w1.cpu().numpy())
            all_w2.append(w2.cpu().numpy())
            all_solute_smiles.extend(solute_smiles)
            all_solv1_smiles.extend(solv1_smiles)
            all_solv2_smiles.extend(solv2_smiles)

            # 收集 M: (B, K, K)
            M_np = M.cpu().numpy()
            ratio_np = w1.cpu().numpy().reshape(-1)  # (B,)
            B_cur = M_np.shape[0]
            for i in range(B_cur):
                s1 = solv1_smiles[i]
                s2 = solv2_smiles[i]
                key = (s1, s2)
                r = float(ratio_np[i])
                # 为了聚类方便，round 一下比例
                r_key = round(r, 3)

                M_dict.setdefault(key, {})
                M_dict[key].setdefault(r_key, [])
                M_dict[key][r_key].append(M_np[i])  # (K,K)

    # 合并 numpy
    all_preds = np.concatenate(all_preds, axis=0).reshape(-1)
    all_labels = np.concatenate(all_labels, axis=0).reshape(-1)
    all_temps = np.concatenate(all_temps, axis=0).reshape(-1)
    all_w1 = np.concatenate(all_w1, axis=0).reshape(-1)
    all_w2 = np.concatenate(all_w2, axis=0).reshape(-1)

    # ---- 测试指标：RMSE + pct<=1 ----
    mse = float(((all_preds - all_labels) ** 2).mean())
    rmse = float(np.sqrt(mse))
    pct1 = pct_within_1_np(all_preds, all_labels)

    print(f"Test RMSE: {rmse:.4f}")
    print(f"Test pct(|err|<=1): {pct1*100:.2f}%")

    # ========================== 画散点图 ==========================
    plt.figure()
    x = all_labels
    y = all_preds
    plt.scatter(x, y, s=15, alpha=0.6)

    # y = x 虚线
    x_line = np.linspace(x.min() - 0.5, x.max() + 0.5, 200)
    plt.plot(x_line, x_line, "k--", linewidth=1.0, label="y = x")

    # +-1 阴影区域
    plt.fill_between(
        x_line,
        x_line - 1.0,
        x_line + 1.0,
        alpha=0.2,
        label="|y - x| <= 1",
    )

    plt.xlabel("True logS")
    plt.ylabel("Predicted logS")
    plt.title(f"Test RMSE={rmse:.3f}, pct<=1={pct1*100:.1f}%")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "test_scatter.png"), dpi=300)
    plt.close()

    # =================== 保存测试集全部预测结果 ===================
    df_pred = pd.DataFrame(
        {
            "solute_smiles": all_solute_smiles,
            "solvent1_smiles": all_solv1_smiles,
            "solvent2_smiles": all_solv2_smiles,
            "Xsolvent1": all_w1,
            "Xsolvent2": all_w2,
            "temperature": all_temps,
            "logS_true": all_labels,
            "logS_pred": all_preds,
            "abs_err": np.abs(all_preds - all_labels),
        }
    )
    df_pred.to_csv(os.path.join(out_dir, "test_predictions.csv"), index=False)

    # =================== 每一对溶剂在不同比例下的 M 热力图 ===================
    # M_dict[(s1,s2)][ratio] -> [ (K,K), (K,K), ... ]
    for (s1, s2), ratio_map in M_dict.items():
        # 文件名前处理一下，避免特殊字符
        def _safe(s: str) -> str:
            return (
                s.replace("/", "_")
                .replace("\\", "_")
                .replace(":", "_")
                .replace("*", "")
                .replace("?", "")
                .replace("<", "")
                .replace(">", "")
                .replace("|", "")
            )

        s1_safe = _safe(s1)
        s2_safe = _safe(s2)

        for r_key, mats in ratio_map.items():
            mats_np = np.stack(mats, axis=0)  # (N,K,K)
            mean_M = mats_np.mean(axis=0)     # (K,K)

            plt.figure()
            plt.imshow(mean_M, origin="lower", aspect="equal")
            plt.colorbar(label="M value")
            K = mean_M.shape[0]
            plt.xticks(range(K), [f"sp{j}" for j in range(K)])
            plt.yticks(range(K), [f"sp{i}" for i in range(K)])
            plt.xlabel("solvent2 ChemSpaces (B)")
            plt.ylabel("solvent1 ChemSpaces (A)")
            plt.title(f"{s1}\n{s2}\nXsolvent1≈{r_key:.3f}")

            plt.tight_layout()
            fname = f"{s1_safe}__{s2_safe}__x1_{r_key:.3f}.png"
            plt.savefig(os.path.join(m_dir, fname), dpi=300)
            plt.close()

    np.savez(
        os.path.join(out_dir, "training_stats.npz"),
        train_losses=np.array(train_losses),
        val_losses=np.array(val_losses),
        test_rmse=rmse,
        test_pct_le1=pct1,
    )

    return {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "test_rmse": rmse,
        "test_pct_le1": pct1,
    }


In [14]:
 ckpt_paths = ["ckpt/tau_0.1__scale_0.25/task_0/best_model.pth","ckpt/tau_0.1__scale_0.25/task_1/best_model.pth","ckpt/tau_0.1__scale_0.25/task_2/best_model.pth","ckpt/tau_0.1__scale_0.25/task_none/best_model.pth"]

In [15]:
def run_all_mix_experiments(num_runs: int = 3):
    """
    Run 4 experiments, each with `num_runs` repetitions:

      Splits:
        S1: train/val/test = 0.3 / 0.1 / 0.6
        S2: train/val/test = 0.8 / 0.1 / 0.1

      For each split S:
        - Experiment A: use_attention = True   (tag 'att')
        - Experiment B: use_attention = False  (tag 'noatt')

    对于同一轮 run_idx 和同一 split：
      attention / no-attention 共用完全相同的 train/val/test 划分
      （通过相同 random_state 和同一组 loader 实现）

    对每个实验（比如 'att_0.3_0.1_0.6'），统计:
      - 3 次 test_rmse 的均值 / 方差
      - 3 次 pct(|err|<=1) 的均值 / 方差

    结果目录结构：
      mix_result/
        att_0.3_0.1_0.6/
          run_1/
          run_2/
          run_3/
        noatt_0.3_0.1_0.6/
          ...
        att_0.8_0.1_0.1/
        noatt_0.8_0.1_0.1/
    """
    base_out_dir = "mix_result_solv"
    ensure_dir(base_out_dir)

    # 两种数据划分配置
    split_configs = [
        ("0.3_0.1_0.6", 0.3, 0.1, 0.6),
        ("0.8_0.1_0.1", 0.8, 0.1, 0.1),
    ]

    # 记录所有实验的结果
    # all_stats[exp_name] = {
    #     "runs": [stats_run1, stats_run2, stats_run3],
    #     "rmse": [..],
    #     "pct":  [..],
    #     "rmse_mean": float,
    #     "rmse_var":  float,
    #     "pct_mean":  float,
    #     "pct_var":   float,
    # }
    all_stats = {}

    for split_name, train_frac, val_frac, test_frac in split_configs:
        print("\n" + "=" * 80)
        print(f"Data split: train/val/test = {train_frac}/{val_frac}/{test_frac}")
        print("=" * 80)

        for run_idx in range(num_runs):
            print(f"\n--- Split {split_name}, Run {run_idx+1}/{num_runs} ---")

            # 这一轮 run 的 random_state
            cur_seed = 42 + run_idx

            # 本轮：构建一次数据集划分
            train_loader, val_loader, test_loader = make_mixture_loaders_by_solvent_pair(
                csv_path="mix_noaq.csv",
                batch_size=64,
                train_frac=train_frac,
                val_frac=val_frac,
                test_frac=test_frac,
                random_state=cur_seed,
            )

            # 在同一 data split + run_idx 下，对 att / noatt 分别训练
            for use_attention in (True, False):
                tag = "att" if use_attention else "noatt"
                exp_name = f"{tag}_{split_name}"

                print(
                    f"[{exp_name}] Run {run_idx+1}/{num_runs} "
                    f"(use_attention={use_attention}, seed={cur_seed})"
                )

                # 若第一次遇到该实验，初始化容器
                if exp_name not in all_stats:
                    all_stats[exp_name] = {
                        "runs": [],
                        "rmse": [],
                        "pct":  [],
                    }

                # 1) 构建模型（注意要有 use_attention 参数）
                mix_model = build_mix_model_subset(
                    sample_s=train_loader.dataset[0][0],  # just used to probe dims
                    sample_v=train_loader.dataset[0][1],   # just used to probe dims
                    ckpt_paths=ckpt_paths,
                    solute_dropout=0.1,
                    solvent_dropout=0.1,
                    mlp_hidden=(512, 512),
                    mlp_dropout=0.1,
                    t_embed_dim=16,
                    use_entmax=False,
                    ratio_dim=1,
                    ratio_hidden_dim=16,
                    use_attention=use_attention,
                )

                # 2) 加载单溶剂预训练参数
                mix_model = load_pretrained_into_bmix(
                    mix_model,
                    ckpt_path="ckpt/leeds.pth",
                    device=device,
                    freeze_encoders=True,
                    freeze_fusions=True,
                    freeze_combiner=False,
                )

                # 3) 训练 + 测试（单次）
                out_dir = os.path.join(base_out_dir, exp_name, f"run_{run_idx+1}")
                ensure_dir(out_dir)

                stats = train_and_eval_mix_model(
                    model=mix_model,
                    train_loader=train_loader,
                    val_loader=val_loader,
                    test_loader=test_loader,
                    num_epochs=100,
                    lr=1e-4,
                    out_dir=out_dir,
                )

                # 记录本次 run 的结果
                all_stats[exp_name]["runs"].append(stats)
                all_stats[exp_name]["rmse"].append(stats["test_rmse"])
                all_stats[exp_name]["pct"].append(stats["test_pct_le1"])

    # ========== 统计每个实验的均值和方差，并打印 ==========
    print("\n" + "=" * 80)
    print("Summary over experiments (mean / var across runs)")
    print("=" * 80)

    for exp_name, rec in all_stats.items():
        rmse_arr = np.array(rec["rmse"], dtype=float)
        pct_arr  = np.array(rec["pct"], dtype=float)

        rmse_mean = float(rmse_arr.mean())
        rmse_var  = float(rmse_arr.var())
        pct_mean  = float(pct_arr.mean())
        pct_var   = float(pct_arr.var())

        rec["rmse_mean"] = rmse_mean
        rec["rmse_var"]  = rmse_var
        rec["pct_mean"]  = pct_mean
        rec["pct_var"]   = pct_var

        print(f"\n[Experiment: {exp_name}]")
        print(f"  RMSE runs      : {rmse_arr}")
        print(f"  RMSE mean / var: {rmse_mean:.4f} / {rmse_var:.6f}")
        print(f"  pct<=1 runs    : {pct_arr}")
        print(f"  pct<=1 mean/var: {pct_mean*100:.2f}% / {pct_var*10000:.4f} (in %^2)")

    return all_stats


In [17]:
# ============================================================
# Mixture ablation test (ATT mode) using your saved Leeds ablation checkpoints
# Split: train/val/test = 0.3 / 0.1 / 0.6 (by solvent pair)
#
# IMPORTANT:
# - This cell ASSUMES you already ran the code (BMixSolvationPredictor,
#   build_mix_model_subset, make_mixture_loaders_by_solvent_pair, train_and_eval_mix_model, etc.)
# - We WILL load the corresponding ablation checkpoint:
#     ablation_solvation/subset_{name}/model_best.pth
#   into the BMix model BEFORE training/evaluating on mixture data.
# ============================================================

import os
import copy
import numpy as np
import pandas as pd
import torch

# -----------------------------
# 0) user config (edit these)
# -----------------------------
MIX_CSV = "mix_noaq.csv"

# Your saved Leeds ablation folder (from the screenshot)
ABL_DIR = "ablation_solvation"

# Pretrain base dir used to build per-space encoder ckpts (same structure as your ablation code)
# (Needed only to instantiate encoders; weights will be overwritten by Leeds checkpoints.)
PRETRAIN_BASE_DIR = "ckpt/tau_0.1__scale_0.25"
TASK_DIRS = ["task_0", "task_1", "task_2", "task_none"]  # 4 spaces

# mixture split (what you asked)
TRAIN_FRAC = 0.3
VAL_FRAC   = 0.1
TEST_FRAC  = 0.6
RANDOM_STATE = 2025
BATCH_SIZE = 64

# training hyperparams on mixture (only for the mixture stage)
MIX_EPOCHS = 80
MIX_LR     = 1e-3

# Build BMix model hyperparams MUST match your leeds ablation training setup
SOLUTE_DROPOUT  = 0.3
SOLVENT_DROPOUT = 0.3
MLP_HIDDEN      = (1024, 128, 64)
MLP_DROPOUT     = 0.2
T_EMBED_DIM     = 32

# ATT mode only
USE_ATTENTION = True

# Recommended: freeze the loaded leeds parts, and only train the new residual/att modules.
# If you want to finetune everything on mixture, set these to False.
FREEZE_ENCODERS = True
FREEZE_FUSIONS  = True
FREEZE_COMBINER = True


# -----------------------------
# 1) helpers
# -----------------------------
def ensure_dir(p: str):
    os.makedirs(p, exist_ok=True)

def pct_within_1_np(preds: np.ndarray, labels: np.ndarray) -> float:
    preds = preds.reshape(-1)
    labels = labels.reshape(-1)
    return float((np.abs(preds - labels) <= 1.0).mean())

def _unwrap_subset_dataset(ds):
    # handle torch.utils.data.Subset nesting
    from torch.utils.data import Subset
    while isinstance(ds, Subset):
        ds = ds.dataset
    return ds

def get_sample_graphs_from_mixture_loader(train_loader):
    base_ds = _unwrap_subset_dataset(train_loader.dataset)
    # MixtureDataset.__getitem__ returns:
    # (solute_graph, solvent1_graph, solvent2_graph, temp, label, frac1, frac2, smiles...)
    sample = base_ds[0]
    sample_s = sample[0]    # solute graph
    sample_v = sample[1]    # solvent1 graph (just for inferring input dims)
    return sample_s, sample_v

def make_ckpt_paths_all(pretrain_base_dir: str):
    ckpts = []
    for sub in TASK_DIRS:
        ckpt = os.path.join(pretrain_base_dir, sub, "best_model.pth")
        if not os.path.isfile(ckpt):
            raise FileNotFoundError(f"Missing pretrain encoder ckpt: {ckpt}")
        ckpts.append(ckpt)
    return ckpts

def load_leeds_ablation_into_bmix(
    model,
    leeds_ckpt_path: str,
    device: str = "cpu",
    freeze_encoders: bool = True,
    freeze_fusions: bool = True,
    freeze_combiner: bool = True,
):
    """
    Load your saved leeds ablation checkpoint (SolvationPredictor) into BMixSolvationPredictor.

    Your saved file: ablation_solvation/subset_xxx/model_best.pth
    contains:
        {"model_state_dict": ... , ...}

    Keys match BMix for:
        sol_encs.*, solv_encs.*, sol_f.*, solv_f.*, combine.*
    BMix-specific params (attends/cross_scale/space_head_*) are not in ckpt -> remain random init.
    """
    ckpt = torch.load(leeds_ckpt_path, map_location=device)
    state = ckpt.get("model_state_dict", ckpt)

    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"[load_leeds_ablation_into_bmix] loaded: {leeds_ckpt_path}")
    print("  missing:", missing)
    print("  unexpected:", unexpected)

    freeze_prefixes = []
    if freeze_encoders:
        freeze_prefixes += ["sol_encs.", "solv_encs."]
    if freeze_fusions:
        freeze_prefixes += ["sol_f.", "solv_f."]
    if freeze_combiner:
        freeze_prefixes += ["combine."]

    n_frozen = 0
    for n, p in model.named_parameters():
        if any(n.startswith(pre) for pre in freeze_prefixes):
            p.requires_grad = False
            n_frozen += 1
    print(f"[load_leeds_ablation_into_bmix] frozen params: {n_frozen}")
    return model


# -----------------------------
# 2) build mixture loaders (0.3/0.1/0.6, by solvent pair)
# -----------------------------
train_loader, val_loader, test_loader = make_mixture_loaders_by_solvent_pair(
    MIX_CSV,
    batch_size=BATCH_SIZE,
    train_frac=TRAIN_FRAC,
    val_frac=VAL_FRAC,
    test_frac=TEST_FRAC,
    random_state=RANDOM_STATE,
    canonicalize_pair=True,
)

sample_s, sample_v = get_sample_graphs_from_mixture_loader(train_loader)
ckpt_paths_all = make_ckpt_paths_all(PRETRAIN_BASE_DIR)

# -----------------------------
# 3) subsets (3-of-4)
# -----------------------------
SUBSETS = {
    "0_1_2_3": [0, 1, 2, 3],
}

# output root for this mixture ablation run
OUT_ROOT = f"mix_abl_att_split_{TRAIN_FRAC}_{VAL_FRAC}_{TEST_FRAC}"
ensure_dir(OUT_ROOT)

# -----------------------------
# 4) run ablation: for each subset
#    - build BMix (ATT)
#    - load the corresponding leeds ablation checkpoint
#    - train on mixture loaders
#    - eval & save
# -----------------------------
rows = []

for subset_name, idxs in SUBSETS.items():
    print("\n" + "=" * 90)
    print(f"[Mixture Ablation | ATT] subset={subset_name} idxs={idxs}  split={TRAIN_FRAC}/{VAL_FRAC}/{TEST_FRAC}")
    print("=" * 90)

    # ---- build subset ckpts for instantiation ----
    ckpt_paths_sub = [ckpt_paths_all[i] for i in idxs]

    # ---- build BMix model (ATT mode) ----
    model = build_mix_model_subset(
        sample_s=sample_s,
        sample_v=sample_v,
        ckpt_paths=ckpt_paths_sub,
        solute_dropout=SOLUTE_DROPOUT,
        solvent_dropout=SOLVENT_DROPOUT,
        mlp_hidden=MLP_HIDDEN,
        mlp_dropout=MLP_DROPOUT,
        t_embed_dim=T_EMBED_DIM,
        use_entmax=False,
        ratio_dim=1,
        ratio_hidden_dim=16,
        use_attention=USE_ATTENTION,  # ATT only
    )

    # ---- load the corresponding Leeds ablation model ----
    leeds_ckpt = os.path.join(ABL_DIR, f"subset_{subset_name}", "model_best.pth")
    if not os.path.isfile(leeds_ckpt):
        raise FileNotFoundError(f"Cannot find leeds ablation checkpoint for subset {subset_name}: {leeds_ckpt}")

    model = load_leeds_ablation_into_bmix(
        model,
        leeds_ckpt_path=leeds_ckpt,
        device=str(device),
        freeze_encoders=FREEZE_ENCODERS,
        freeze_fusions=FREEZE_FUSIONS,
        freeze_combiner=FREEZE_COMBINER,
    )

    # ---- train/eval on mixture (this uses YOUR existing train_and_eval_mix_model) ----
    out_dir = os.path.join(OUT_ROOT, f"subset_{subset_name}")
    stats = train_and_eval_mix_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        num_epochs=MIX_EPOCHS,
        lr=MIX_LR,
        out_dir=out_dir,
    )

    rows.append({
        "subset": subset_name,
        "idxs": "_".join(map(str, idxs)),
        "split": f"{TRAIN_FRAC}/{VAL_FRAC}/{TEST_FRAC}",
        "mix_test_rmse": stats["test_rmse"],
        "mix_test_pct_le1": stats["test_pct_le1"],   # already in [0,1]
        "leeds_init_ckpt": leeds_ckpt,
        "out_dir": out_dir,
        "freeze_encoders": FREEZE_ENCODERS,
        "freeze_fusions": FREEZE_FUSIONS,
        "freeze_combiner": FREEZE_COMBINER,
        "use_attention": USE_ATTENTION,
    })

# -----------------------------
# 5) save summary
# -----------------------------
df = pd.DataFrame(rows).sort_values("mix_test_rmse", ascending=True)
summary_path = os.path.join(OUT_ROOT, "summary_mix_ablation_att.csv")
df.to_csv(summary_path, index=False)

print("\n✅ Done. Summary saved to:", summary_path)
df



[Mixture Ablation | ATT] subset=0_1_2_3 idxs=[0, 1, 2, 3]  split=0.3/0.1/0.6


c:\Users\27476\Documents\GitHub\ChemSpaceAlign\finetune_pair.py:383: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ck = torch.load(path, map_location=device)
C:\Users\27476\

[load_leeds_ablation_into_bmix] loaded: ablation_solvation\subset_0_1_2_3\model_best.pth
  missing: ['cross_scale', 'sol_encs.0.mpn.encoder.cached_zero_vector', 'sol_encs.0.mpn.encoder.W_i.weight', 'sol_encs.0.mpn.encoder.W_i.bias', 'sol_encs.0.mpn.encoder.W_h.weight', 'sol_encs.0.mpn.encoder.W_h.bias', 'sol_encs.0.mpn.encoder.W_o.weight', 'sol_encs.0.mpn.encoder.W_o.bias', 'sol_encs.0.projection_head.0.weight', 'sol_encs.0.projection_head.0.bias', 'sol_encs.0.projection_head.3.weight', 'sol_encs.0.projection_head.3.bias', 'sol_encs.1.mpn.encoder.cached_zero_vector', 'sol_encs.1.mpn.encoder.W_i.weight', 'sol_encs.1.mpn.encoder.W_i.bias', 'sol_encs.1.mpn.encoder.W_h.weight', 'sol_encs.1.mpn.encoder.W_h.bias', 'sol_encs.1.mpn.encoder.W_o.weight', 'sol_encs.1.mpn.encoder.W_o.bias', 'sol_encs.1.projection_head.0.weight', 'sol_encs.1.projection_head.0.bias', 'sol_encs.1.projection_head.3.weight', 'sol_encs.1.projection_head.3.bias', 'sol_encs.2.mpn.encoder.cached_zero_vector', 'sol_encs.2.m

,subset,idxs,split,mix_test_rmse,mix_test_pct_le1,leeds_init_ckpt,out_dir,freeze_encoders,freeze_fusions,freeze_combiner,use_attention
0,0_1_2_3,0_1_2_3,0.3/0.1/0.6,1.307428,0.564322,ablation_solvation\subset_0_1_2_3\model_best.pth,mix_abl_att_split_0.3_0.1_0.6\subset_0_1_2_3,True,True,True,True


In [ ]:
all_stats = run_all_mix_experiments()

In [ ]:
all_stats